# SCIO-Bench: Anomaly Detection for Off-Grid Solar-Battery IoT

**Universitas Jenderal Soedirman — Karel Tsalasatir Riyan — April 2026**

This notebook orchestrates the full SCIO-Bench benchmark pipeline:
- Phase 1: Dataset download & preprocessing
- Phase 2: Synthetic variable augmentation (SOC, voltage, current)
- Phase 3: Anomaly injection → SCIO-Bench labeled dataset
- Phase 4: Feature engineering & train/val/test split
- Phase 5: Rule-Based Threshold (Method A)
- Phase 6: Isolation Forest + LOF (Method B)
- Phase 7: LSTM Autoencoder (Method C — L1)
- Phase 8: Hierarchical L2 (Random Forest + SMOTE)
- Phase 9: XAI — SHAP + Reconstruction Error Analysis
- Phase 10: Edge Hardware Profiling + TFLite INT8
- Phase 11: Statistical Tests + Results Tables
- Phase 12: Visualizations (5 figures)
- Phase 13: Final Export

All `random_state=42`. All splits are **chronological** (no data leakage).

## Environment Setup

Run this cell first if on Google Colab.

In [ ]:
# ─── Google Colab setup (skip if running locally) ───────────────────────────
import os
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    # Install dependencies
    !pip install tensorflow scikit-learn pandas numpy matplotlib seaborn shap scipy imbalanced-learn joblib kaggle pyod -q

    # Upload kaggle.json for dataset download
    from google.colab import files
    print("Upload your kaggle.json file:")
    files.upload()
    !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

    # Clone repo (if not already done)
    if not os.path.exists('src'):
        !git clone https://github.com/<username>/scio-anomaly-benchmark.git .
else:
    print('Running locally — ensure requirements are installed via: pip install -r requirements.txt')

In [ ]:
# ─── Global imports & constants ──────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.abspath('.'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Global random seed — DO NOT CHANGE for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Output directories
os.makedirs('outputs/dataset', exist_ok=True)
os.makedirs('outputs/results', exist_ok=True)
os.makedirs('outputs/figures', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)
os.makedirs('data/splits', exist_ok=True)

print('Environment ready.')
print(f'NumPy {np.__version__} | Pandas {pd.__version__}')

---
## Phase 1 — Dataset Download & Preprocessing

In [ ]:
# Phase 1 — implemented in src/data/download.py and src/data/preprocess.py
# TODO: Implemented in Phase 1
from src.data.download import download_kaggle_dataset
from src.data.preprocess import load_and_preprocess

download_kaggle_dataset(dest='data/raw/')
df_plant1, df_plant2 = load_and_preprocess('data/raw/')
print(f'Plant 1: {df_plant1.shape} | Plant 2: {df_plant2.shape}')

---
## Phase 2 — Synthetic Variable Augmentation

In [ ]:
# Phase 2 — implemented in src/data/augmentation.py
# TODO: Implemented in Phase 2
from src.data.augmentation import augment_dataset

df_plant1 = augment_dataset(df_plant1, plant_id='PLANT1')
df_plant2 = augment_dataset(df_plant2, plant_id='PLANT2')
print('Augmented columns:', df_plant1.columns.tolist())

---
## Phase 3 — Anomaly Injection (SCIO-Bench)

In [ ]:
# Phase 3 — implemented in src/data/anomaly_injection.py
# TODO: Implemented in Phase 3
from src.data.anomaly_injection import inject_all_anomalies

df_bench = inject_all_anomalies(df_plant1, df_plant2, random_seed=RANDOM_SEED)
df_bench.to_csv('outputs/dataset/scio_bench_dataset.csv', index=False)
print(df_bench['anomaly_type'].value_counts())

---
## Phase 4 — Feature Engineering & Data Splitting

In [ ]:
# Phase 4 — implemented in src/data/feature_engineering.py
# TODO: Implemented in Phase 4
from src.data.feature_engineering import engineer_features, split_chronological

df_feat = engineer_features(df_bench)
train, val, test = split_chronological(df_feat)
print(f'Train: {train.shape} | Val: {val.shape} | Test: {test.shape}')

---
## Phase 5 — Method A: Rule-Based Threshold

In [ ]:
# Phase 5 — implemented in src/models/rule_based.py
# TODO: Implemented in Phase 5
from src.models.rule_based import RuleBasedDetector
from src.evaluation.metrics import compute_all_metrics

rb = RuleBasedDetector()
rb.fit(train)
y_pred_rb = rb.predict(test)
metrics_rb = compute_all_metrics(test, y_pred_rb, method_name='Rule-Based')
print(metrics_rb)

---
## Phase 6 — Method B: Isolation Forest + LOF

In [ ]:
# Phase 6 — implemented in src/models/isolation_forest.py and lof.py
# TODO: Implemented in Phase 6
from src.models.isolation_forest import IFDetector
from src.models.lof import LOFDetector

if_model = IFDetector(random_state=RANDOM_SEED)
if_model.fit(train, val)
y_pred_if = if_model.predict(test)
metrics_if = compute_all_metrics(test, y_pred_if, method_name='Isolation Forest')

lof_model = LOFDetector()
lof_model.fit(train, val)
y_pred_lof = lof_model.predict(test)
metrics_lof = compute_all_metrics(test, y_pred_lof, method_name='LOF')

---
## Phase 7 — Method C: LSTM Autoencoder (L1)

In [ ]:
# Phase 7 — implemented in src/models/lstm_autoencoder.py
# TODO: Implemented in Phase 7
from src.models.lstm_autoencoder import LSTMAutoencoder

lstm_ae = LSTMAutoencoder(random_state=RANDOM_SEED)
lstm_ae.fit(train, val)
y_pred_lstm, recon_errors = lstm_ae.predict(test)
metrics_lstm = compute_all_metrics(test, y_pred_lstm, method_name='LSTM AE')

---
## Phase 8 — Hierarchical L2 Classifier

In [ ]:
# Phase 8 — implemented in src/models/hierarchical_l2.py
# TODO: Implemented in Phase 8
from src.models.hierarchical_l2 import HierarchicalL2

l2 = HierarchicalL2(random_state=RANDOM_SEED)
l2.fit(train, lstm_ae)
y_pred_l2 = l2.predict(test, y_pred_lstm)

# Alarm-budgeted evaluation
from src.evaluation.metrics import alarm_budgeted_eval
mir10 = alarm_budgeted_eval(recon_errors, test['is_anomaly'], budget_per_day=10)
print(f'MIR@10: {mir10:.3f}')

---
## Phase 9 — XAI Analysis

In [ ]:
# Phase 9 — implemented in src/xai/
# TODO: Implemented in Phase 9
from src.xai.shap_analysis import compute_shap
from src.xai.reconstruction_analysis import per_feature_reconstruction_error

shap_values = compute_shap(if_model.model, test)
feature_errors = per_feature_reconstruction_error(lstm_ae, test)

---
## Phase 10 — Edge Hardware Profiling

In [ ]:
# Phase 10 — implemented in src/evaluation/edge_profiling.py
# TODO: Implemented in Phase 10
from src.evaluation.edge_profiling import profile_all_models, quantize_to_tflite

edge_table = profile_all_models(
    models={'Rule-Based': rb, 'IF': if_model, 'LOF': lof_model, 'LSTM AE': lstm_ae},
    X_test=test, n_runs=1000
)
tflite_metrics = quantize_to_tflite(lstm_ae, test)
print(edge_table)

---
## Phase 11 — Statistical Tests & Results Tables

In [ ]:
# Phase 11 — implemented in src/evaluation/
# TODO: Implemented in Phase 11
from src.evaluation.statistical_tests import mcnemar_test

stat_rb_if = mcnemar_test(y_pred_rb, y_pred_if, test['is_anomaly'])
stat_if_lstm = mcnemar_test(y_pred_if, y_pred_lstm, test['is_anomaly'])

# Compile result tables
all_metrics = [metrics_rb, metrics_if, metrics_lof, metrics_lstm]
table1 = pd.DataFrame([m['f1_per_type'] for m in all_metrics], index=['Rule-Based','IF','LOF','LSTM AE'])
table1.to_csv('outputs/results/table1_f1_comparison.csv')
print(table1)

---
## Phase 12 — Visualizations

In [ ]:
# Phase 12 — implemented in src/visualization/plots.py
# TODO: Implemented in Phase 12
from src.visualization.plots import (
    plot_roc_curves, plot_timeseries_example,
    plot_f1_bar, plot_shap_summary, plot_recon_error
)

plot_roc_curves(all_metrics, save_path='outputs/figures/fig1_roc_curves.pdf')
plot_timeseries_example(df_bench, save_path='outputs/figures/fig2_timeseries_example.pdf')
plot_f1_bar(table1, save_path='outputs/figures/fig3_f1_per_anomaly_type.pdf')
plot_shap_summary(shap_values, save_path='outputs/figures/fig4_shap_isolation_forest.pdf')
plot_recon_error(feature_errors, save_path='outputs/figures/fig5_recon_error_lstm.pdf')

---
## Phase 13 — Final Export

In [ ]:
print('=== SCIO-Bench Pipeline Complete ===')
print('Outputs:')
for root, dirs, files in os.walk('outputs'):
    for f in files:
        print(f'  {os.path.join(root, f)}')